# DeBERTa pred

Run this notebook from `german/`. Dataset and adapter paths refer to local files.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, AutoConfig
from peft import PeftModel, PeftConfig
import torch
from torch.utils.data import DataLoader
import polars as pl


In [ ]:
device = torch.device('cuda:0')
torch_dtype = torch.bfloat16


In [ ]:
peft_config = PeftConfig.from_pretrained("./results2/checkpoint-22516")


In [ ]:
config = AutoConfig.from_pretrained(peft_config.base_model_name_or_path)
base_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    config=config,
    torch_dtype=torch.bfloat16
)


In [ ]:
model = PeftModel.from_pretrained(base_model, "./results2/checkpoint-22516")
model.to(device)
model.eval()


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path, use_fast=True)


In [ ]:
from datasets import Dataset
df = pl.read_csv("../data/eng_processed_test.tsv", separator="\t", quote_char=None)
print(len(df))
test_set = Dataset.from_polars(df)
print(len(test_set))


In [ ]:
class TokenizedDataset(Dataset):
    def __init__(self, dataset, tokenizer):
        self.dataset = dataset
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        tokenized = self.tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )
        return {
            "input_ids": tokenized["input_ids"].squeeze(0),
            "attention_mask": tokenized["attention_mask"].squeeze(0)
        }

test_dataset = TokenizedDataset(test_set, tokenizer)


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

predictions = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        predictions.extend(preds)

with open("predictions2.tsv", "w") as f:
    f.write("id\tlabel\n")
    for i, pred in enumerate(predictions):
        f.write(f"{test_set[i]['id']}\t{pred}\n")


In [ ]:
predictions = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = torch.nn.functional.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        predictions.extend(probs)

with open("predictions_probabilities.tsv", "w") as f:
    f.write("id\tprobability\n")
    for i, prob in enumerate(predictions):
        f.write(f"{test_set[i]['id']}\t{prob[1]}\n")


In [ ]:
predictions = []
probabilities = []
threshold = 0.4755

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = torch.nn.functional.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        probabilities.extend(probs)
        preds = (probs[:, 1] >= threshold).astype(int)
        predictions.extend(preds)

with open("predictions2_thres.tsv", "w") as f:
    f.write("id\tlabel\n")
    for i, pred in enumerate(predictions):
        f.write(f"{test_set[i]['id']}\t{pred}\n")
